<img src='https://hammondm.github.io/hltlogo1.png' style="float:right">

LING 593A-011<br>
Fall 2025<br>
Davo Acevedo-Cardona

# GUU — Firebase Upload Tool

This notebook is a test sample to upload all the CSV to firebase.

Sai, Lindsey and I still need to agree where the data will be uploaded (Firebase project).


# Imports

In [1]:
from __future__ import annotations
from ftfy import fix_text
import base64
import csv
import json
import os
import re
import datetime as dt
from collections import Counter
from dataclasses import dataclass
from email.utils import parsedate_to_datetime
from typing import Iterable, List, Optional, Tuple
from tqdm.auto import tqdm

try:
    from rapidfuzz import fuzz, process
    HAVE_RAPIDFUZZ = True
except Exception:
    HAVE_RAPIDFUZZ = False

# Class Configuration

In [2]:
@dataclass
class Config:
    gmail_query: str = 'subject:"New Company Request via iOS" before:2025/01/01'
    max_messages: Optional[int] = None

    checkpoint_path: str = "brand_email_checkpoint.jsonl"
    extracted_csv: str = "brand_request_extracted.csv"
    counts_csv: str = "brand_request_counts.csv"
    alias_review_csv: str = "alias_review.csv"

    auto_merge_threshold: int = 90
    review_threshold_low: int = 80
    evidence_max_chars: int = 160


cfg = Config(
    gmail_query='(subject:"New Company Request via iOS" OR subject:"New Company Request via Android") before:2025/01/01',
    max_messages=None,
)


# API Auth

In [3]:
def build_gmail_service(
    credentials_json_path: str = "credentials.json",
    token_path: str = "token.json"
):
    from googleapiclient.discovery import build
    from google_auth_oauthlib.flow import InstalledAppFlow
    from google.auth.transport.requests import Request
    from google.oauth2.credentials import Credentials

    SCOPES = ["https://www.googleapis.com/auth/gmail.readonly"]
    creds = None

    if os.path.exists(token_path):
        creds = Credentials.from_authorized_user_file(token_path, SCOPES)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                credentials_json_path, SCOPES
            )
            creds = flow.run_local_server(port=0)
        with open(token_path, "w", encoding="utf-8") as f:
            f.write(creds.to_json())

    return build("gmail", "v1", credentials=creds)

# Fetch

In [4]:
def list_message_ids(service, user_id: str, query: str, max_messages: Optional[int]):
    ids = []
    page_token = None

    while True:
        resp = service.users().messages().list(
            userId=user_id,
            q=query,
            pageToken=page_token,
            maxResults=500
        ).execute()

        for m in resp.get("messages", []) or []:
            ids.append(m["id"])
            if max_messages and len(ids) >= max_messages:
                return ids

        page_token = resp.get("nextPageToken")
        if not page_token:
            break

    return ids


def get_message(service, user_id: str, message_id: str):
    return service.users().messages().get(
        userId=user_id, id=message_id, format="full"
    ).execute()


def get_header(headers, name: str) -> str:
    for h in headers:
        if h.get("name", "").lower() == name.lower():
            return h.get("value", "")
    return ""

# Email Body Decoding

In [5]:
def decode_b64url(data: str) -> str:
    if not data:
        return ""
    pad = len(data) % 4
    if pad:
        data += "=" * (4 - pad)
    return base64.urlsafe_b64decode(data).decode("utf-8", errors="replace")


def extract_plaintext(payload: dict) -> str:
    def walk(p):
        parts = [p]
        for c in p.get("parts", []) or []:
            parts.extend(walk(c))
        return parts

    for part in walk(payload):
        if part.get("mimeType") == "text/plain":
            return decode_b64url(part.get("body", {}).get("data", ""))

    return decode_b64url(payload.get("body", {}).get("data", ""))


# Brand Extraction

In [6]:
# -----------------------------
# Cell 6 — Brand Extraction Logic (Core)  [UPDATED for parenthetical lists + "Based in ..."]
# -----------------------------

import re
from typing import List, Optional, Tuple

BRAND_BLOCK_RE = re.compile(
    r"i['’]d\s+like\s+to\s+know\s+more\s+details\s+about:\s*(.*?)\s*thanks",
    flags=re.IGNORECASE | re.DOTALL
)

SUBJECT_FALLBACK_RE = re.compile(
    r"details\s+about:\s*(.*?)\s*(?:thanks|\Z)",
    flags=re.IGNORECASE | re.DOTALL
)

# Trailing junk sometimes included by users after the brand(s)
TRAILING_JUNK_RE = re.compile(
    r"""
    (.*?)                                  # capture main content
    (?:                                    # then drop any of these tails
        (?:\.\s*based\s+in\b.*)$            # ". Based in Portland..."
      | (?:\bbased\s+in\b.*)$              # "Based in Portland..."
      | (?:\blocated\s+in\b.*)$            # "Located in ..."
      | (?:\bbased\s+out\s+of\b.*)$        # "Based out of ..."
    )
    """,
    flags=re.IGNORECASE | re.DOTALL | re.VERBOSE
)

PAREN_LIST_RE = re.compile(r"^\s*(.*?)\s*\((.*?)\)\s*$", flags=re.DOTALL)

def extract_brand_block(body: str, subject: str) -> Tuple[Optional[str], str]:
    if body:
        m = BRAND_BLOCK_RE.search(body)
        if m:
            raw = m.group(1).strip()
            return (raw if raw else None), "body"

    if subject:
        m = SUBJECT_FALLBACK_RE.search(subject)
        if m:
            raw = m.group(1).strip()
            return (raw if raw else None), "subject"

    return None, "none"


def _split_commas_outside_parens(s: str) -> List[str]:
    out, buf, depth = [], [], 0
    for ch in s:
        if ch == "(":
            depth += 1
        elif ch == ")":
            depth = max(0, depth - 1)

        if ch == "," and depth == 0:
            piece = "".join(buf).strip()
            if piece:
                out.append(piece)
            buf = []
        else:
            buf.append(ch)

    last = "".join(buf).strip()
    if last:
        out.append(last)
    return out


def split_brands(raw: str) -> List[str]:
    raw = (raw or "").strip()
    if not raw:
        return []

    # 0) Remove trailing location/descriptor junk (e.g., ". Based in Portland Oregon")
    m_junk = TRAILING_JUNK_RE.match(raw)
    if m_junk:
        raw = (m_junk.group(1) or "").strip()

    # 1) Split on slashes first
    parts = []
    for chunk in raw.split("/"):
        chunk = re.sub(r"\s+", " ", chunk).strip()
        if chunk:
            parts.append(chunk)

    # 2) Split on commas outside parentheses
    tmp = []
    for p in parts:
        tmp.extend(_split_commas_outside_parens(p))
    parts = [re.sub(r"\s+", " ", p).strip() for p in tmp if p.strip()]

    # 3) Expand parenthetical lists:
    #    "Columbia Sportswear (Sorel, Prana, Mountain Hardware)"
    #    -> ["Columbia Sportswear", "Sorel", "Prana", "Mountain Hardware"]
    expanded = []
    for p in parts:
        pm = PAREN_LIST_RE.match(p)
        if pm:
            main = pm.group(1).strip()
            inside = pm.group(2).strip()
            if main:
                expanded.append(main)

            # split inside on commas (safe because we're inside parentheses)
            inside_items = [x.strip() for x in inside.split(",") if x.strip()]
            expanded.extend(inside_items)
        else:
            expanded.append(p)

    # 4) Conservative split on "and" (case-insensitive, safe)
    exceptions = {
        "wine and spirits",
        "oil and gas",
        "research and development",
    }
    generic_tails = {"spirits", "vodka", "tequila", "store", "stores", "company"}

    final = []
    for p in expanded:
        p = re.sub(r"\s+", " ", p).strip()
        low = p.lower()

        if any(exc in low for exc in exceptions):
            final.append(p)
            continue

        if re.search(r"\s+and\s+", p, flags=re.IGNORECASE):
            and_parts = re.split(r"\s+and\s+", p, maxsplit=1, flags=re.IGNORECASE)
            if len(and_parts) == 2:
                left, right = and_parts[0].strip(), and_parts[1].strip()
                if right.lower() in generic_tails:
                    final.append(p)
                else:
                    if left:
                        final.append(left)
                    if right:
                        final.append(right)
            else:
                final.append(p)
        else:
            final.append(p)

    # final cleanup
    return [x for x in final if x.strip()]


# Normalization & Fuzzy Canonicalization

In [7]:
QUESTION_PREFIX_RE = re.compile(r"^\s*(do|does|is|are|can|could|would|should)\b", re.I)
COMMENTARY_RE = re.compile(r"\b(im assuming|i think|republican|democrat|democrats|politics|support)\b", re.I)
WEBSITE_RE = re.compile(r"\b(website\s+is|site\s+is|www\.|https?://|\.com\b|\.net\b|\.org\b)\b", re.I)
ALIAS_MAP = {
    "Cosco": "Costco",
    "Humanna": "Humana",
    "Safeways": "Safeway",
    "Sketchers": "Skechers",
    "Dominoes": "Domino's",
    "Tommy'S Car Wash": "Tommy's Car Wash",
    "Pendelton": "Pendleton",
}

def apply_alias(clean: str) -> str:
    return ALIAS_MAP.get(clean, clean)

def clean_candidate_brand(raw: str) -> str | None:
    """
    Returns a cleaned brand candidate or None if it should be dropped.
    """
    if not raw:
        return None

    s = re.sub(r"\s+", " ", raw).strip()
    s_low = s.lower()

    # If it’s clearly a "Website is ..." type line, drop it
    if WEBSITE_RE.search(s):
        # But keep if it's just a bare domain-like brand? (optional)
        return None

    # If it looks like a question/commentary, try to salvage by taking the last segment after '?'
    if "?" in s:
        tail = s.split("?")[-1].strip()
        # If tail is short, treat it as the brand
        if 2 <= len(tail) <= 60:
            s = tail
            s_low = s.lower()
        else:
            return None

    # Drop if it starts like a question
    if QUESTION_PREFIX_RE.search(s_low):
        return None

    # Drop if it contains explicit political commentary
    if COMMENTARY_RE.search(s_low):
        # Try salvage: if there is a period, take last clause
        if "." in s:
            tail = s.split(".")[-1].strip()
            if 2 <= len(tail) <= 60 and not COMMENTARY_RE.search(tail.lower()):
                s = tail
            else:
                return None
        else:
            return None

    # Drop obvious generic nouns/categories (expand as you review)
    GENERIC = {
        "gas", "bank", "paper towel", "paper towels", "paper products",
        "toilet paper", "hair accessories", "a yarn store"
    }
    if s_low in GENERIC:
        return None

    # If it’s too long (likely a sentence), drop
    if len(s) > 80:
        return None

    # If it has too many words, probably not a brand
    if len(s.split()) > 8:
        return None

    return s
    
def normalize_brand(s: str) -> str:
    s = fix_text(s)
    s = re.sub(r"[.!?;:\-]+$", "", s.strip())
    s = re.sub(r"\s+", " ", s)

    if re.fullmatch(r"[A-Z0-9&]{2,}", s):
        return s  # acronym

    return s.title()


def fuzzy_canonicalize(brand: str, canon: Iterable[str], cfg: Config):
    if not HAVE_RAPIDFUZZ:
        return brand, None

    match = process.extractOne(
        brand, list(canon), scorer=fuzz.token_set_ratio
    )

    if not match:
        return brand, None

    best, score, _ = match

    if score >= cfg.auto_merge_threshold:
        return best, None

    if cfg.review_threshold_low <= score < cfg.auto_merge_threshold:
        return brand, {
            "brand_raw": brand,
            "suggested_canonical": best,
            "score": score
        }

    return brand, None


# Checkpointing & CSV Writers

In [8]:
def load_processed_ids(path: str) -> set:
    seen = set()
    if not os.path.exists(path):
        return seen

    with open(path, encoding="utf-8") as f:
        for line in f:
            try:
                seen.add(json.loads(line)["message_id"])
            except Exception:
                pass
    return seen


def append_checkpoint(path: str, message_id: str):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps({"message_id": message_id}) + "\n")


def write_csv(path: str, rows: list, fields: list):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        w.writerows(rows)

# Loop

In [9]:
# -----------------------------
# Cell 9 — Main Extraction Loop (with progress bar + ETA)
# -----------------------------

def run(cfg: Config):
    service = build_gmail_service()

    # Load checkpoint
    processed = load_processed_ids(cfg.checkpoint_path)

    # Fetch message IDs
    msg_ids = list_message_ids(
        service,
        "me",
        cfg.gmail_query,
        cfg.max_messages
    )

    # Only process unprocessed messages (so progress bar is accurate)
    to_process = [mid for mid in msg_ids if mid not in processed]

    print(f"Found {len(msg_ids)} messages.")
    print(f"Skipping {len(msg_ids) - len(to_process)} already processed.")
    print(f"Processing {len(to_process)} messages.\n")

    extracted = []
    review = []

    counts = Counter()
    first_seen = {}
    last_seen = {}

    pbar = tqdm(
        to_process,
        total=len(to_process),
        desc="Processing brand request emails",
        unit="email",
        dynamic_ncols=True
    )

    for mid in pbar:
        msg = get_message(service, "me", mid)
        payload = msg.get("payload", {})
        headers = payload.get("headers", [])

        subject = get_header(headers, "Subject")
        sender = get_header(headers, "From")
        date_raw = get_header(headers, "Date")

        try:
            msg_dt = parsedate_to_datetime(date_raw)
        except Exception:
            internal_ms = int(msg.get("internalDate", "0"))
            msg_dt = dt.datetime.fromtimestamp(
                internal_ms / 1000.0,
                tz=dt.timezone.utc
            )

        body = extract_plaintext(payload)

        block, source = extract_brand_block(body, subject)
        if not block:
            append_checkpoint(cfg.checkpoint_path, mid)
            continue

        canon_set = set(counts.keys())

        for raw in split_brands(block):
            raw2 = clean_candidate_brand(raw)
            if not raw2:
                continue
            clean = normalize_brand(raw2)
            clean = apply_alias(clean)

            # Exact match first
            canonical = None
            for c in canon_set:
                if c.lower() == clean.lower():
                    canonical = c
                    break

            if canonical is None:
                canonical, rev = fuzzy_canonicalize(clean, canon_set, cfg)
                if rev:
                    review.append(rev)
            else:
                rev = None

            counts[canonical] += 1
            canon_set.add(canonical)

            first_seen[canonical] = min(
                first_seen.get(canonical, msg_dt), msg_dt
            )
            last_seen[canonical] = max(
                last_seen.get(canonical, msg_dt), msg_dt
            )

            extracted.append({
                "message_id": mid,
                "date": msg_dt.isoformat(),
                "from": sender,
                "subject": subject,
                "brand_raw": raw,
                "brand_clean": clean,
                "brand_canonical": canonical,
                "source": source
            })

        append_checkpoint(cfg.checkpoint_path, mid)

        # Live progress info
        pbar.set_postfix(
            rows=len(extracted),
            brands=len(counts)
        )

    # -----------------------------
    # Write outputs
    # -----------------------------

    write_csv(
        cfg.extracted_csv,
        extracted,
        [
            "message_id",
            "date",
            "from",
            "subject",
            "brand_raw",
            "brand_clean",
            "brand_canonical",
            "source"
        ]
    )

    write_csv(
        cfg.counts_csv,
        [
            {
                "brand_canonical": brand,
                "count": cnt,
                "first_seen": first_seen[brand].isoformat(),
                "last_seen": last_seen[brand].isoformat()
            }
            for brand, cnt in counts.most_common()
        ],
        ["brand_canonical", "count", "first_seen", "last_seen"]
    )

    if review:
        write_csv(
            cfg.alias_review_csv,
            review,
            ["brand_raw", "suggested_canonical", "score"]
        )

    print("\nRun complete.")
    print(f"Total extracted rows: {len(extracted)}")
    print(f"Unique brands: {len(counts)}")

# Run

In [11]:
run(cfg)

Found 320642 messages.
Skipping 0 already processed.
Processing 320642 messages.



Processing brand request emails:   0%|                                                   | 0/320642 [00:00<?, …


Run complete.
Total extracted rows: 195458
Unique brands: 35286
